In [ ]:
import pandas as pd

In [ ]:
df= pd.read_csv('/content/100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [ ]:
#token

def tokenizer(text):
  text=text.lower()
  text=text.replace(".","")
  text=text.replace(",","")
  text=text.replace("?","")
  text=text.replace("!","")
  return text.split()

In [ ]:
tokenizer("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [ ]:
#Vocab
vocab={'UNK':0}
def build_vocab(row):
  tokenize_ques=tokenizer(row['question'])
  tokenize_ans=tokenizer(row['answer'])
  merge=tokenize_ques+tokenize_ans
  for word in merge:
    if word not in vocab:
      vocab[word]=len(vocab)


In [ ]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [ ]:
vocab

{'UNK': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 "'to": 12,
 'kill': 13,
 'a': 14,
 "mockingbird'": 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 "'1984'": 67,
 'george-orwell': 68,
 'currency': 69,
 'un

In [ ]:
# convert words to numerical indecis
def text_to_indices(text,vocab):
  indexed_text=[]
  for token in tokenizer(text):
    if token not in vocab:
      indexed_text.append(vocab['UNK'])
    else:
      indexed_text.append(vocab[token])
  return indexed_text

In [ ]:
import torch
from torch.utils.data import Dataset,DataLoader

In [ ]:
class QADataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab
  def __len__(self):
    return len(self.df)
  def __getitem__(self,idx):
    row=self.df.iloc[idx]
    question=text_to_indices(row['question'],self.vocab)
    answer=text_to_indices(row['answer'],self.vocab)
    return torch.tensor(question),torch.tensor(answer)

In [ ]:
dataset=QADataset(df,vocab)

In [ ]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [ ]:
for question,answer in dataloader:
  print(question,answer)

tensor([[ 42, 257,   2, 258,  83, 259, 260]]) tensor([[261]])
tensor([[ 1,  2,  3, 69,  5, 53]]) tensor([[262]])
tensor([[ 10,  11, 158, 159, 160]]) tensor([[161]])
tensor([[ 10, 310,   3, 311, 312]]) tensor([[313]])
tensor([[ 10, 140,   3, 141, 172,   5,   3,  70, 173]]) tensor([[174]])
tensor([[ 10,  29, 130, 131]]) tensor([[132]])
tensor([[10,  2,  3, 66,  5, 67]]) tensor([[68]])
tensor([[10, 29,  3, 30, 31]]) tensor([[32]])
tensor([[  1,   2,   3,   4,   5, 135]]) tensor([[136]])
tensor([[ 78,  79, 290,  81,  19,  14, 291]]) tensor([[85]])
tensor([[  1,   2,   3,  92, 137,  19,   3,  45]]) tensor([[186]])
tensor([[ 42, 292, 293, 118, 294, 159, 295, 296]]) tensor([[297]])
tensor([[ 1,  2,  3, 24, 25,  5, 26, 19, 27]]) tensor([[28]])
tensor([[ 42, 301, 302, 118,  14, 303, 304, 159, 305, 306, 307, 308]]) tensor([[309]])
tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([[36]])
tensor([[10, 75, 76]]) tensor([[77]])
tensor([[  1,   2,   3,  17, 115,  83,  84]]) tensor([[116]])
tensor([[ 1, 

In [ ]:
import torch.nn as nn

In [ ]:
class SimpleRNN(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64,batch_first=True)
    self.fc=nn.Linear(64,vocab_size)
  def forward(self,question):
    embeded_question=self.embedding(question)
    hidden,final=self.rnn(embeded_question)
    output=self.fc(final.squeeze(0))
    return output

In [ ]:
learning_rate=0.001
epochs=20

In [ ]:
model=SimpleRNN(len(vocab))
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [ ]:
# training loop
for epoch in range(epochs):
  total_loss=0
  for question,answer in dataloader:
    optimizer.zero_grad()
    #forward pass
    output=model(question)
    # loss
    loss=loss_fn(output,answer[0])
    # gradients
    loss.backward()
    #update
    optimizer.step()
    total_loss+=loss.item()
  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 523.414264
Epoch: 2, Loss: 451.505186
Epoch: 3, Loss: 372.017580
Epoch: 4, Loss: 313.409217
Epoch: 5, Loss: 261.993646
Epoch: 6, Loss: 214.285829
Epoch: 7, Loss: 171.732493
Epoch: 8, Loss: 134.515727
Epoch: 9, Loss: 103.720474
Epoch: 10, Loss: 80.490430
Epoch: 11, Loss: 62.973930
Epoch: 12, Loss: 49.888226
Epoch: 13, Loss: 40.221608
Epoch: 14, Loss: 33.129421
Epoch: 15, Loss: 27.396166
Epoch: 16, Loss: 23.008432
Epoch: 17, Loss: 19.442821
Epoch: 18, Loss: 16.699340
Epoch: 19, Loss: 14.336977
Epoch: 20, Loss: 12.407089


In [ ]:
def predict(model,question,threshold=0.5):
  # conver question to numbers
  numerical_question= text_to_indices(question,vocab)
  #tensor
  question_tensor=torch.tensor(numerical_question).unsqueeze(0)
  #send to model
  output=model(question_tensor)
  #probabilitys
  probs=torch.nn.functional.softmax(output,dim=1)
  value,index=torch.max(probs,dim=1)
  if value<threshold:
    print("I do not know")
  else:
    print(list(vocab.keys())[index])


In [ ]:
predict(model,"what is the capital of Bangladesh?")

I do not know
